In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
anotherdata = pd.read_csv(our_data + "ginnascores.txt", sep="\t", header=0)

In [ ]:
result = anotherdata["complex_name"].str.split(r"_", expand=True)
result.columns = ["enzyme", "substrate", "cc"]
result["substrate_cc"] = result["substrate"] + result["cc"]

anotherdata = pd.concat([anotherdata, result], axis=1)

ourdata = pd.read_pickle(join(CURRENT_DIR, "..", "data", "our_data", "5foldsdata.pkl"))

In [ ]:
g_sub = anotherdata["substrate"].unique()
e_sub = ourdata["substrate"].unique()

In [12]:
elements_only_in_substrate_cc = np.setdiff1d(g_sub, e_sub)

In [13]:
elements_only_in_substrate_cc

array(['08Y', '0GV', '0QA', '12H', '1CA', '225', '2IO', '2UO', '3QZ',
       '4UH', '7PF', '88L', 'ANN', 'ASD', 'AUI', 'B9O', 'B9R', 'C8B',
       'CAM', 'CGM', 'CLR', 'CNL', 'D2V', 'DCR', 'DXJ', 'EPD', 'EV1',
       'EY5', 'FLI', 'JZ3', 'K2B', 'KEQ', 'LPH', 'MBN', 'MIV', 'MY8',
       'MYR', 'NCT', 'NRB', 'PAM', 'PLM', 'Q4M', 'REA', 'RRM', 'SI5',
       'SNE', 'STR', 'TES', 'VD3', 'VGJ', 'XAN', 'XBK', 'ZMP'],
      dtype=object)

In [15]:
e_sub

array(['ABA', 'ABI', 'ABS', 'ABT', 'ACA', 'ADI', 'AGI', 'AHT', 'ALK',
       'ARA', 'BAM', 'BIS', 'BRS', 'BSI', 'BYB', 'CAA', 'CAL', 'CAP',
       'CAR', 'CAS', 'CAT', 'CAV', 'CBC', 'CDC', 'CHE', 'CHL', 'CHP',
       'CHR', 'CIN', 'CIT', 'CLA', 'COL', 'COS', 'COU', 'CSL', 'CTH',
       'CTN', 'CUC', 'Cholesterol', 'DAA', 'DAI', 'DAMM', 'DCA', 'DCT',
       'DCU', 'DDE', 'DEA', 'DEM', 'DHC', 'DID', 'DIH', 'DIT', 'DIY',
       'DOA', 'DOC', 'DOS', 'DPR', 'DXN', 'EAC', 'EAH', 'ECAS', 'EGE',
       'EHA', 'EIYA', 'EJI', 'ELO', 'ELS', 'ELY', 'EPI', 'ERI', 'ESA',
       'ESBO', 'ESO', 'ETH', 'EZI', 'FAC', 'FCC', 'FER', 'FEU', 'FKA',
       'FOR', 'FRA', 'FRI', 'GAD', 'GEA', 'GEI', 'GEN', 'GER', 'GEW',
       'GGE', 'GHQ', 'GIA', 'GLC', 'Glucobrassicin', 'HAB', 'HAC', 'HBM',
       'HBOA', 'HCA', 'HCHO', 'HCO', 'HCU', 'HJI', 'HME', 'HMT', 'HNE',
       'HOA', 'HOAN', 'HOX', 'HPD', 'HPOT13', 'HPT', 'HSU', 'HTH', 'HYD',
       'HYO', 'HYR', 'Hexahomomethionine', 'IAM', 'IAO', 'ICN', 'IMD',
    

In [ ]:
elements_only_in_substrate_cc = np.setdiff1d(e_sub, g_sub)

In [18]:
elements_only_in_substrate_cc

array(['Cholesterol', 'DAMM', 'ECAS', 'EIYA', 'ESBO', 'Glucobrassicin',
       'HBOA', 'HCHO', 'HOAN', 'HPOT13', 'Hexahomomethionine', 'INAC',
       'Lanosterol', 'Liquiritigenin', 'MCAR', 'MYI', 'NO', 'OLAC', 'OLE',
       'PHL', 'ZEAA', 'Zeinoxanthin', 'carlactone', 'copalol',
       'fluometuron', 'manool', 'miltiradiene', 'naringenin', 'nicotine'],
      dtype=object)

In [19]:
g_sub

array(['08Y', '0GV', '0QA', '12H', '1CA', '225', '2IO', '2UO', '3QZ',
       '4UH', '7PF', '88L', 'ABA', 'ABI', 'ABS', 'ABT', 'ALK', 'ANN',
       'ARA', 'ASD', 'AUI', 'B9O', 'B9R', 'BAM', 'BIS', 'BRS', 'BSI',
       'BYB', 'C8B', 'CAA', 'CAL', 'CAM', 'CAP', 'CAR', 'CAS', 'CAT',
       'CAV', 'CBC', 'CDC', 'CGM', 'CHE', 'CHL', 'CHP', 'CHR', 'CIN',
       'CIT', 'CLA', 'CLR', 'CNL', 'COL', 'COS', 'COU', 'CSL', 'CTH',
       'CTN', 'CUC', 'D2V', 'DAA', 'DAI', 'DCA', 'DCR', 'DCT', 'DCU',
       'DDE', 'DEA', 'DEM', 'DHC', 'DID', 'DIH', 'DIT', 'DIY', 'DOA',
       'DOC', 'DOS', 'DPR', 'DXJ', 'DXN', 'EAC', 'EAH', 'EGE', 'EHA',
       'EJI', 'ELO', 'ELS', 'ELY', 'EPD', 'EPI', 'ERI', 'ESA', 'ESO',
       'ETH', 'EV1', 'EZI', 'FAC', 'FCC', 'FER', 'FEU', 'FKA', 'FOR',
       'FRA', 'GAD', 'GEA', 'GEI', 'GEN', 'GER', 'GEW', 'GGE', 'GHQ',
       'GIA', 'GLC', 'HAB', 'HAC', 'HBM', 'HCA', 'HCU', 'HJI', 'HME',
       'HMT', 'HNE', 'HOA', 'HOX', 'HPD', 'HPT', 'HSU', 'HTH', 'HYD',
       'HYO', 'HYR',